 Runners

A runner decides how a call is executed and where outputs go. You pick it once; the rest of the code doesn't change.

| Runner | Runs tools... | Use when |
|---|---|---|
| `LocalRunner` (default) | installed on this machine | FSL/ANTs/... are on your `PATH` |
| `DockerRunner` | in the tool's Docker image | laptops, workstations |
| `SingularityRunner` | in an Apptainer/Singularity image | HPC clusters |
| `DryRunner` | nowhere, only builds the command | checking commands, no Docker |

In [ ]:
import logging
import shlex
import shutil
from pathlib import Path

import niwrap
from niwrap import fsl

T1W = "../data/ds000001/sub-01/anat/sub-01_T1w.nii.gz"
WORK = Path("../work/02")

RUNNER = "auto"  # change to "dry" if Docker doesn't work on your machine

# Local (the default)

Without setup, NiWrap runs tools installed on your machine.

In [ ]:
runner = niwrap.get_global_runner()
type(runner).__name__, runner.data_dir

If FSL isn't installed, that fails:

In [ ]:
niwrap.use_local(data_dir=WORK)
try:
    fsl.bet(infile=T1W)
    print("FSL is installed locally, bet ran.")
except FileNotFoundError as e:
    print("No local FSL:", e)

# Docker in one line

After `niwrap.use_docker()`, each call runs in its tool's image (`brainlife/fsl`, `antsx/ants`, ...), pulled on first use. Outputs go to `data_dir`.

In [ ]:
runner = niwrap.use_docker(data_dir=WORK)
type(runner).__name__

On a cluster: `niwrap.use_singularity(...)`. Podman: `niwrap.use_podman(...)`.

# DryRunner: see commands without running them

Commands are built and validated, nothing runs. `last_cargs` holds the last command.

In [ ]:
runner = niwrap.use_dry()
bet = fsl.bet(infile=T1W, binary_mask=True)
print(shlex.join(runner.last_cargs))
bet.outfile

Outputs still have names, so a whole pipeline can be dry-run. The files are never created.

Any single call can also take its own runner (`runner=...`, as in notebook 01).

# Let NiWrap pick

`use_auto()` takes the first available of Docker, Podman, Apptainer/Singularity, then local tools, so one script runs on a laptop and on a cluster. `resolve_runner()` shows what it will pick:

In [ ]:
niwrap.resolve_runner()

In [ ]:
runner = niwrap.use_dry() if RUNNER == "dry" else niwrap.use_auto(data_dir=WORK)
bet = fsl.bet(infile=T1W, binary_mask=True)

# Where are my files?

Each call gets a folder in `data_dir` named `<session id>_<call number>_<tool>`. `bet.root` points to it.

In [ ]:
print(bet.root)
if RUNNER != "dry":
    print(sorted(p.name for p in bet.root.iterdir()))

Container runners also write the command to `run.sh`. Treat `data_dir` as scratch: copy what you keep.

In [ ]:
keep = Path("../derivatives/02/sub-01_desc-brain_mask.nii.gz")
if RUNNER != "dry":
    keep.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(bet.binary_mask, keep)
    print(keep)
# shutil.rmtree(runner.data_dir)  # deletes ALL outputs of this session

# Quieter logs

Runners log each command and the tool's output, which for ANTs registration is thousands of lines. At `INFO` you get one line per call:

In [ ]:
if RUNNER != "dry":
    logging.getLogger(runner.logger_name).setLevel(logging.INFO)
fsl.fslreorient2std(input_image=T1W, output_image="t1w_reoriented.nii.gz").output_image